In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from pathlib import Path


import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.applications import MobileNetV2, DenseNet121
from tensorflow.keras.preprocessing import image
from sklearn.metrics import classification_report, confusion_matrix


print("TensorFlow version:", tf.__version__)


# Utility: set seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [2]:
import os

BASE_DIR = "/kaggle/input/lungs-disease-dataset-4-types/Lung Disease Dataset"  # <-- CHANGE this to the actual dataset path
train_dir = os.path.join(BASE_DIR, "train")
val_dir   = os.path.join(BASE_DIR, "val")
# If you don't have a separate val directory, we'll split training later

IMG_SIZE = (224, 224)  # MobileNetV2 default input size; DenseNet121 also handles this
BATCH_SIZE = 32
NUM_CLASSES = None  # we'll infer

# quick checks
for d in [BASE_DIR, train_dir, val_dir]:
    print(d, "exists?", os.path.exists(d))


/kaggle/input/lungs-disease-dataset-4-types/Lung Disease Dataset exists? True
/kaggle/input/lungs-disease-dataset-4-types/Lung Disease Dataset/train exists? True
/kaggle/input/lungs-disease-dataset-4-types/Lung Disease Dataset/val exists? True


In [ ]:
# Corrected loader for dataset structured as:
# data_root/
#    train/
#       CLASS_A/
#       CLASS_B/
#    val/
#       CLASS_A/
#       CLASS_B/
#    test/
#       CLASS_A/
#       CLASS_B/
#
# This cell will load file lists from each split and build tf.data datasets.

import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

# ---------- CONFIG ----------
ROOT = "/kaggle/input/lungs-disease-dataset-4-types"   # top-level
# it was previously detected as containing "Lung Disease Dataset" wrapper
# set data_root to that folder
data_root = os.path.join(ROOT, "Lung Disease Dataset") if os.path.isdir(os.path.join(ROOT, "Lung Disease Dataset")) else ROOT

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

print("Using data_root =", data_root)
print("Top-level inside data_root:", sorted(os.listdir(data_root)))

# ---------- Detect if train/val/test exist ----------
train_dir = os.path.join(data_root, "train")
val_dir   = os.path.join(data_root, "val")
test_dir  = os.path.join(data_root, "test")

# Helper to list class subfolders inside a split
def list_classes_in_split(split_dir):
    if not os.path.exists(split_dir):
        return []
    return sorted([d for d in os.listdir(split_dir) if os.path.isdir(os.path.join(split_dir, d)) and not d.startswith('.')])

if os.path.exists(train_dir) and os.path.exists(val_dir) and os.path.exists(test_dir):
    print("Detected explicit 'train', 'val', and 'test' folders (good).")
    # Get class names from train folder (authoritative)
    class_names = list_classes_in_split(train_dir)
    if not class_names:
        raise FileNotFoundError(f"No class subfolders found inside {train_dir}. Each split should contain class folders.")
    print("Detected classes from train/:", class_names)

    # Function to gather file paths + labels for a given split dir
    def gather_paths_and_labels(split_dir):
        filepaths = []
        labels = []
        for idx, cls in enumerate(class_names):
            cls_path = os.path.join(split_dir, cls)
            if not os.path.isdir(cls_path):
                # if a class folder is missing in this split, skip or warn
                print(f"Warning: class folder missing in {split_dir}: {cls}")
                continue
            for root, _, files in os.walk(cls_path):
                for fname in files:
                    if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')):
                        filepaths.append(os.path.join(root, fname))
                        labels.append(idx)
        return np.array(filepaths), np.array(labels)

    fp_train, y_train = gather_paths_and_labels(train_dir)
    fp_val,   y_val   = gather_paths_and_labels(val_dir)
    fp_test,  y_test  = gather_paths_and_labels(test_dir)

else:
    # No explicit splits present — fall back to previous behavior (single-folder split)
    # We'll attempt to find class subfolders directly under data_root and then split.
    print("No explicit train/val/test found — falling back to single-folder split if possible.")
    class_names = sorted([d for d in os.listdir(data_root) if os.path.isdir(os.path.join(data_root, d)) and not d.startswith('.')])
    if not class_names:
        raise FileNotFoundError("No class subfolders found under data_root.")
    print("Detected classes:", class_names)

    # gather all filepaths and labels
    filepaths = []
    labels = []
    for idx, cls in enumerate(class_names):
        cls_dir = os.path.join(data_root, cls)
        for root, _, files in os.walk(cls_dir):
            for fname in files:
                if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')):
                    filepaths.append(os.path.join(root, fname))
                    labels.append(idx)
    filepaths = np.array(filepaths); labels = np.array(labels)
    # stratified split: 80% train, then remaining 20% -> 10% val, 10% test
    fp_train, fp_rem, y_train, y_rem = train_test_split(filepaths, labels, test_size=0.2, stratify=labels, random_state=SEED)
    fp_val, fp_test, y_val, y_test = train_test_split(fp_rem, y_rem, test_size=0.5, stratify=y_rem, random_state=SEED)

# ---------- Print counts ----------
print("Counts -> train:", len(fp_train), "val:", len(fp_val), "test:", len(fp_test))

# ---------- Build tf.data datasets ----------
def load_and_preprocess(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.convert_image_dtype(image, tf.float32)
    image = tf.image.resize(image, IMG_SIZE)
    image = mobilenet_preprocess(image * 255.0)
    label = tf.one_hot(label, depth=len(class_names))
    return image, label

def build_ds(filepaths_arr, labels_arr, training=False):
    ds = tf.data.Dataset.from_tensor_slices((filepaths_arr, labels_arr))
    if training:
        ds = ds.shuffle(buffer_size=min(10000, len(filepaths_arr)), seed=SEED)
    ds = ds.map(lambda p, l: load_and_preprocess(p, l), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = build_ds(fp_train, y_train, training=True)
val_ds   = build_ds(fp_val,   y_val,   training=False)
test_ds  = build_ds(fp_test,  y_test,  training=False)

NUM_CLASSES = len(class_names)
print("NUM_CLASSES =", NUM_CLASSES)
print("class_names =", class_names)
print("Train batches:", int(tf.data.experimental.cardinality(train_ds).numpy()))
print("Val batches:  ", int(tf.data.experimental.cardinality(val_ds).numpy()))
print("Test batches: ", int(tf.data.experimental.cardinality(test_ds).numpy()))

# ---------- GPU check (explain cuInit message) ----------
gpus = tf.config.list_physical_devices('GPU')
print("GPUs detected by TF:", gpus)
if not gpus:
    print("Note: No GPU detected. The cuInit message is informational — TF will run on CPU.")
else:
    print("GPU(s) present — if you still see cuInit errors, check CUDA/driver compatibility.")


In [ ]:
# Corrected augmentation + application cell
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# Ensure AUTOTUNE exists (set it if missing)
try:
    AUTOTUNE
except NameError:
    AUTOTUNE = tf.data.AUTOTUNE

# Data augmentation block (Keras preprocessing layers)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.05),
], name="data_augmentation")

# NOTE:
# - If your train_ds images are already preprocessed to match MobileNetV2 (i.e. mobilenet_preprocess applied),
#   augmentation layers can be applied directly (they accept float images).
# - If your train_ds images are raw uint8 [0,255], you'd want to apply augmentation BEFORE any scaling.
#
# The loader I provided earlier already applied mobilenet_preprocess inside load_and_preprocess,
# so this code assumes images are floats in the range expected by your model.

def apply_augmentation(ds):
    """
    Apply augmentation to a dataset of (image, label) pairs.
    This maps augmentation only on the image tensor.
    """
    def _augment(image, label):
        # Keras augmentation layers expect rank-4 tensor for batches; they handle rank-3 too.
        image = data_augmentation(image)
        return image, label
    return ds.map(_augment, num_parallel_calls=AUTOTUNE)

# Apply augmentation only to training dataset
train_ds_aug = apply_augmentation(train_ds)
# Keep validation/test unchanged (no augmentation)
val_ds = val_ds
test_ds = test_ds

# Optional: visualize a small batch to confirm augmentation visually
def show_batch(dataset, class_names, n=9):
    batch = next(iter(dataset))
    images, labels = batch
    images = images.numpy()
    labels = labels.numpy()
    plt.figure(figsize=(7,7))
    for i in range(min(n, images.shape[0])):
        plt.subplot(3,3,i+1)
        # images may be in [-1,1] if mobilenet_preprocess was applied; scale to [0,1] for display
        img = images[i]
        if img.min() < -0.5 and img.max() <= 1.0:
            img_disp = (img + 1.0) / 2.0
        else:
            img_disp = img
        plt.imshow(img_disp)
        lbl_idx = int(labels[i].argmax())
        plt.title(class_names[lbl_idx], fontsize=9)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

# Use class_names inferred earlier (from loader)
try:
    class_names
except NameError:
    # fallback: try to get from train_ds (if it was an image_dataset_from_directory)
    try:
        class_names = train_ds.class_names
    except Exception:
        class_names = [str(i) for i in range(NUM_CLASSES)]

print("Showing one augmented train batch (visual check)...")
show_batch(train_ds_aug, class_names)


In [ ]:
def select_feature_layers_by_shape(model, input_size, target_scales=(4, 8, 16, 32)):
    """
    Automatically selects feature map layers from a CNN backbone
    whose output shapes correspond to certain downscaling ratios
    (useful for FPN feature selection).

    Args:
        model: tf.keras.Model (e.g., MobileNetV2 or DenseNet121, include_top=False)
        input_size: tuple (H, W) — model input resolution, e.g., (224, 224)
        target_scales: tuple of integers
            Spatial downscaling factors relative to input image.
            Example: scale=4 -> feature map of size H/4 × W/4

    Returns:
        dict: {scale_factor: layer_name} for layers that match requested scales
    """
    h, w = input_size
    shapes = {}

    for layer in model.layers:
        # Only consider layers with 4D feature maps (batch, H, W, C)
        if hasattr(layer, 'output') and hasattr(layer.output, 'shape'):
            out_shape = layer.output.shape
            if len(out_shape) == 4:
                oh = int(out_shape[1]) if out_shape[1] is not None else None
                ow = int(out_shape[2]) if out_shape[2] is not None else None
                if oh and ow:
                    ratio_h = h // oh
                    ratio_w = w // ow
                    # Check if same scale in both dimensions and matches target
                    if ratio_h == ratio_w and ratio_h in target_scales:
                        shapes[ratio_h] = layer.name

    return shapes


In [ ]:
# ============================================================
# Create MobileNetV2 & DenseNet121 feature extraction backbones
# ============================================================

import tensorflow as tf
from tensorflow.keras import models

# Helper to get layer output safely
def get_layer_output(model, layer_name):
    """Returns the output tensor of a layer if it exists, else None."""
    try:
        return model.get_layer(layer_name).output
    except Exception as e:
        print(f"Warning: Layer {layer_name} not found in {model.name}: {e}")
        return None

# -----------------------------
# Create base pretrained models
# -----------------------------
backbone_mobilenet = tf.keras.applications.MobileNetV2(
    include_top=False, weights='imagenet', input_shape=(224, 224, 3)
)
backbone_densenet = tf.keras.applications.DenseNet121(
    include_top=False, weights='imagenet', input_shape=(224, 224, 3)
)

# Assuming you've already run select_feature_layers_by_shape()
# to get mob_shapes and dense_shapes dictionaries
# (If not, run select_feature_layers_by_shape(backbone_mobilenet, (224,224)) etc.)

# Example (in case you didn’t define them yet)
try:
    mob_shapes
except NameError:
    from pprint import pprint
    mob_shapes = select_feature_layers_by_shape(backbone_mobilenet, (224, 224))
    dense_shapes = select_feature_layers_by_shape(backbone_densenet, (224, 224))
    print("Auto-detected MobileNet shapes:"); pprint(mob_shapes)
    print("Auto-detected DenseNet shapes:"); pprint(dense_shapes)

# -----------------------------
# Choose layers for FPN fusion
# -----------------------------
c1_name = mob_shapes.get(4)
c2_name = mob_shapes.get(8)
c3_name = mob_shapes.get(16)
c4_name = dense_shapes.get(16)
c5_name = dense_shapes.get(32)

# Fallbacks if automatic mapping fails
mob_layer_names = [l.name for l in backbone_mobilenet.layers]
dense_layer_names = [l.name for l in backbone_densenet.layers]

if not c1_name and 'block_1_expand_relu' in mob_layer_names:
    c1_name = 'block_1_expand_relu'
if not c2_name and 'block_3_expand_relu' in mob_layer_names:
    c2_name = 'block_3_expand_relu'
if not c3_name and 'block_6_expand_relu' in mob_layer_names:
    c3_name = 'block_6_expand_relu'

if not c4_name:
    # DenseNet examples: 'conv4_block6_concat', 'pool3_conv'
    candidates = [l for l in dense_layer_names if 'conv4' in l or 'pool3' in l]
    c4_name = candidates[-1] if candidates else None
if not c5_name:
    candidates = [l for l in dense_layer_names if 'conv5' in l or 'pool4' in l]
    c5_name = candidates[-1] if candidates else None

# Print chosen layers
print("Chosen feature layers:")
print(f"  C1 (MobileNet): {c1_name}")
print(f"  C2 (MobileNet): {c2_name}")
print(f"  C3 (MobileNet): {c3_name}")
print(f"  C4 (DenseNet):  {c4_name}")
print(f"  C5 (DenseNet):  {c5_name}")

# -----------------------------
# Create functional feature extractors
# -----------------------------
mob_outputs = [get_layer_output(backbone_mobilenet, n) for n in [c1_name, c2_name, c3_name] if n]
dense_outputs = [get_layer_output(backbone_densenet, n) for n in [c4_name, c5_name] if n]

feature_extractor_mob = models.Model(
    inputs=backbone_mobilenet.input,
    outputs=mob_outputs,
    name='mobilenetv2_feats'
)

feature_extractor_den = models.Model(
    inputs=backbone_densenet.input,
    outputs=dense_outputs,
    name='densenet121_feats'
)

# Print model summaries (show only last layers for brevity)
print("\n✅ MobileNetV2 feature extractor summary:")
feature_extractor_mob.summary(line_length=120, expand_nested=False, show_trainable=False)

print("\n✅ DenseNet121 feature extractor summary:")
feature_extractor_den.summary(line_length=120, expand_nested=False, show_trainable=False)


In [ ]:
def fpn_layer(feature_maps, out_channels=128, name_prefix='fpn'):
    lateral = []
    ...
    up = layers.Lambda(lambda x, ref=lateral[i]: resize_to_ref(x, ref))(higher)
    fused = layers.Add()([lateral[i], up])
    ...


In [ ]:
from tensorflow.keras import layers

def gated_attention_module(feature, reduction=16, name_prefix='gated'):
    """
    Combines channel (Squeeze-and-Excitation) and spatial attention mechanisms.

    Args:
        feature: input feature map tensor (H, W, C)
        reduction: reduction ratio for channel attention
        name_prefix: name prefix for layer naming

    Returns:
        Tensor: output after applying gated channel + spatial attention
    """

    # ------------- Channel Gate (Squeeze-and-Excitation) -------------
    ch = feature.shape[-1]
    se = layers.GlobalAveragePooling2D(name=f"{name_prefix}_se_gap")(feature)
    se = layers.Dense(ch // reduction, activation='relu', name=f"{name_prefix}_se_dense1")(se)
    se = layers.Dense(ch, activation='sigmoid', name=f"{name_prefix}_se_dense2")(se)
    se = layers.Reshape((1, 1, ch), name=f"{name_prefix}_se_reshape")(se)
    ch_out = layers.Multiply(name=f"{name_prefix}_se_mul")([feature, se])

    # ------------- Spatial Gate -------------
    spatial = layers.Conv2D(1, kernel_size=1, padding='same', name=f"{name_prefix}_sp_conv")(feature)
    spatial = layers.Activation('sigmoid', name=f"{name_prefix}_sp_sigmoid")(spatial)
    sp_out = layers.Multiply(name=f"{name_prefix}_sp_mul")([feature, spatial])

    # ------------- Combine Channel + Spatial -------------
    out = layers.Add(name=f"{name_prefix}_add")([ch_out, sp_out])
    out = layers.Activation('relu', name=f"{name_prefix}_out_relu")(out)

    return out


In [9]:
# ===== Full corrected model cell (uses ResizeToLike custom layer for robust resizing) =====
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K

# ---------- Config ----------
INPUT_SHAPE = (224, 224, 3)
MODEL_INPUT = tf.keras.Input(shape=INPUT_SHAPE, name='input_image')
NUM_CLASSES = len(class_names) if 'class_names' in globals() else 4

# ---------- Backbones (single input) ----------
backbone_mobilenet = tf.keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_tensor=MODEL_INPUT)
backbone_densenet  = tf.keras.applications.DenseNet121(include_top=False, weights='imagenet', input_tensor=MODEL_INPUT)

# ---------- Utility: select layers by spatial scale ----------
def select_feature_layers_by_shape(model, input_size=(224,224), target_scales=(4,8,16,32)):
    h,w = input_size
    shapes = {}
    for layer in model.layers:
        if hasattr(layer, 'output') and hasattr(layer.output, 'shape'):
            out_shape = layer.output.shape
            if len(out_shape) == 4:
                oh = int(out_shape[1]) if out_shape[1] is not None else None
                ow = int(out_shape[2]) if out_shape[2] is not None else None
                if oh and ow:
                    rh = h // oh
                    rw = w // ow
                    if rh == rw and rh in target_scales:
                        shapes[rh] = layer.name
    return shapes

mob_shapes = select_feature_layers_by_shape(backbone_mobilenet, (INPUT_SHAPE[0], INPUT_SHAPE[1]))
dense_shapes = select_feature_layers_by_shape(backbone_densenet, (INPUT_SHAPE[0], INPUT_SHAPE[1]))

# ---------- Choose layers (with fallbacks) ----------
c1_name = mob_shapes.get(4, 'block_1_expand_relu')
c2_name = mob_shapes.get(8, 'block_3_expand_relu')
c3_name = mob_shapes.get(16, 'block_6_expand_relu')
c4_name = dense_shapes.get(16, 'conv4_block24_concat')
c5_name = dense_shapes.get(32, 'conv5_block16_concat')

def safe_get_tensor(model, layer_name):
    try:
        return model.get_layer(layer_name).output
    except Exception as e:
        print(f"Warning: layer {layer_name} not found in {model.name}: {e}")
        return None

mob_tensors = [safe_get_tensor(backbone_mobilenet, n) for n in (c1_name, c2_name, c3_name)]
den_tensors = [safe_get_tensor(backbone_densenet, n) for n in (c4_name, c5_name)]
all_feats = [t for t in (mob_tensors + den_tensors) if t is not None]

# ---------- Custom layer: ResizeToLike ----------
class ResizeToLike(layers.Layer):
    """Resize 'x' to match spatial HxW of 'ref' using tf.image.resize."""
    def __init__(self, method='nearest', **kwargs):
        super().__init__(**kwargs)
        self.method = method

    def call(self, inputs, **kwargs):
        # inputs expected as tuple/list: (x, ref)
        x, ref = inputs
        ref_shape = tf.shape(ref)[1:3]
        return tf.image.resize(x, ref_shape, method=self.method)

    def compute_output_shape(self, input_shape):
        # input_shape: tuple/list of shapes ((batch, h, w, c_x), (batch, h_ref, w_ref, c_ref))
        x_shape, ref_shape = input_shape
        batch = x_shape[0]
        out_c = x_shape[3]
        # we can't statically determine ref H,W here; return (batch, None, None, channels)
        return (batch, None, None, out_c)

# ---------- Robust FPN using ResizeToLike ----------
def fpn_layer(feature_maps, out_channels=128, name_prefix='fpn'):
    lateral = []
    for i, fm in enumerate(feature_maps):
        lateral.append(layers.Conv2D(out_channels, 1, padding='same', name=f"{name_prefix}_lat_{i}")(fm))

    pyramid = [None] * len(lateral)
    pyramid[-1] = lateral[-1]

    for i in reversed(range(len(lateral) - 1)):
        higher = pyramid[i + 1]
        # Use ResizeToLike to match spatial dims dynamically
        up = ResizeToLike(name=f"{name_prefix}_resize_{i}")([higher, lateral[i]])
        fused = layers.Add(name=f"{name_prefix}_add_{i}")([lateral[i], up])
        pyramid[i] = fused

    # refine with 3x3 conv
    pyramid = [layers.Conv2D(out_channels, 3, padding='same', name=f"{name_prefix}_post_{i}")(p) for i, p in enumerate(pyramid)]
    return pyramid

# ---------- Gated attention module ----------
def gated_attention_module(feature, reduction=8, name_prefix='gated'):
    ch_static = feature.shape[-1]
    # Channel attention
    se = layers.GlobalAveragePooling2D(name=f"{name_prefix}_se_gap")(feature)
    se = layers.Dense(max(1, (ch_static or 128) // reduction), activation='relu', name=f"{name_prefix}_se_dense1")(se)
    se = layers.Dense(ch_static or 128, activation='sigmoid', name=f"{name_prefix}_se_dense2")(se)
    se = layers.Reshape((1,1,ch_static or 128), name=f"{name_prefix}_se_reshape")(se)
    ch_out = layers.Multiply(name=f"{name_prefix}_se_mul")([feature, se])
    # Spatial attention
    spatial = layers.Conv2D(1, 1, padding='same', name=f"{name_prefix}_sp_conv")(feature)
    spatial = layers.Activation('sigmoid', name=f"{name_prefix}_sp_sigmoid")(spatial)
    sp_out = layers.Multiply(name=f"{name_prefix}_sp_mul")([feature, spatial])
    # Combine
    out = layers.Add(name=f"{name_prefix}_add")([ch_out, sp_out])
    out = layers.Activation('relu', name=f"{name_prefix}_out_relu")(out)
    return out

# ---------- Build FPN + gated attention ----------
fpn_feats = fpn_layer(all_feats, out_channels=128, name_prefix='fpn')
gated_feats = [gated_attention_module(p, reduction=8, name_prefix=f'gated_p{i}') for i, p in enumerate(fpn_feats)]

# ---------- Upsample gated levels to highest resolution using ResizeToLike ----------
p0 = gated_feats[0]
upsampled = []
for i, gf in enumerate(gated_feats):
    if i == 0:
        up = gf
    else:
        up = ResizeToLike(name=f'final_resize_{i}')([gf, p0])
        up = layers.Conv2D(128, 3, padding='same', name=f'final_conv_{i}')(up)
        up = layers.BatchNormalization(name=f'final_bn_{i}')(up)
        up = layers.Activation('relu', name=f'final_act_{i}')(up)
    upsampled.append(up)

merged = layers.Concatenate(name='merged_pyramid')(upsampled)

# ---------- Classification head ----------
x = layers.GlobalAveragePooling2D(name='gap')(merged)
x = layers.Dropout(0.5, name='dropout')(x)
activation = 'sigmoid' if NUM_CLASSES == 1 else 'softmax'
units = 1 if NUM_CLASSES == 1 else NUM_CLASSES
output = layers.Dense(units, activation=activation, name='predictions')(x)

model = models.Model(inputs=MODEL_INPUT, outputs=output, name='FPN_GatedAttention_Model')

# ---------- Summary ----------
model.summary(line_length=120)


/tmp/ipykernel_58/49194469.py:11: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  backbone_mobilenet = tf.keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_tensor=MODEL_INPUT)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "FPN_GatedAttention_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━
┃ Layer (type)                      ┃ Output Shape                 ┃           Param # ┃ Connected to              
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━
│ input_image (InputLayer)          │ (None, 224, 224, 3)          │                 0 │ -                         
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ zero_padding2d (ZeroPadding2D)    │ (None, 230, 230, 3)          │                 0 │ input_image[0][0]         
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ conv1_conv (Conv2D)               │ (None, 112, 112, 64)         │             9,408 │ zero_padding2d[0][0]      
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ conv1_bn (BatchNormalization)     │ (None, 112, 112, 64)         │               256 │ conv1_conv[0][0]          
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ conv1_relu (Activation)           │ (None, 112, 112, 64)         │                 0 │ conv1_bn[0][0]            
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ zero_padding2d_1 (ZeroPadding2D)  │ (None, 114, 114, 64)         │                 0 │ conv1_relu[0][0]          
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ pool1 (MaxPooling2D)              │ (None, 56, 56, 64)           │                 0 │ zero_padding2d_1[0][0]    
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ conv2_block1_0_bn                 │ (None, 56, 56, 64)           │               256 │ pool1[0][0]               
│ (BatchNormalization)              │                              │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ conv2_block1_0_relu (Activation)  │ (None, 56, 56, 64)           │                 0 │ conv2_block1_0_bn[0][0]   
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ conv2_block1_1_conv (Conv2D)      │ (None, 56, 56, 128)          │             8,192 │ conv2_block1_0_relu[0][0] 
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ conv2_block1_1_bn                 │ (None, 56, 56, 128)          │               512 │ conv2_block1_1_conv[0][0] 
│ (BatchNormalization)              │                              │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ conv2_block1_1_relu (Activation)  │ (None, 56, 56, 128)          │                 0 │ conv2_block1_1_bn[0][0]   
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ conv2_block1_2_conv (Conv2D)      │ (None, 56, 56, 32)           │            36,864 │ conv2_block1_1_relu[0][0] 
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ conv2_block1_concat (Concatenate) │ (None, 56, 56, 96)           │                 0 │ pool1[0][0],              
│                                   │                              │                   │ conv2_block1_2_conv[0][0] 
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ conv2_block2_0_bn                 │ (None, 56, 56, 96)

 Total params: 9,322,457 (35.56 MB)

 Trainable params: 9,220,505 (35.17 MB)

 Non-trainable params: 101,952 (398.25 KB)

In [ ]:
# Ensure NUM_CLASSES is defined properly
# Try using from dataset (if available)
try:
    NUM_CLASSES = len(class_names)
except Exception:
    NUM_CLASSES = 4  # <-- Fallback (you can adjust this to your dataset)

# Now choose loss safely
loss = 'categorical_crossentropy' if NUM_CLASSES > 1 else 'binary_crossentropy'

# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=loss,
    metrics=['accuracy']
)

# Define callbacks for saving best model and early stopping
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath='/kaggle/working/best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=4,
        restore_best_weights=True,
        verbose=1
    )
]

print(f"✅ Model compiled successfully with {NUM_CLASSES} output classes and loss = '{loss}'")


In [ ]:
EPOCHS = 8
history = model.fit(
train_ds,
validation_data=val_ds,
epochs=EPOCHS,
callbacks=callbacks
)

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# ---------- 1. Evaluate on validation set ----------
val_loss, val_acc = model.evaluate(val_ds, verbose=1)
print(f"\n✅ Validation loss: {val_loss:.4f} | Validation accuracy: {val_acc:.4f}\n")

# ---------- 2. Predict on validation set ----------
y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1).tolist())
    y_pred.extend(np.argmax(preds, axis=1).tolist())

# ---------- 3. Classification report ----------
print("📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# ---------- 4. (Optional) Confusion Matrix ----------
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:\n", cm)

# If you want to visualize it nicely:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix - Validation Set")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.preprocessing import image

# ---------------------------------------------------------
# 1. LOAD THE MODEL
# ---------------------------------------------------------
model_path = "/kaggle/working/best_model.keras"

print(f"Loading model from: {model_path}")
model = load_model(model_path, compile=True, custom_objects={"ResizeToLike": ResizeToLike})
print("✅ Model loaded successfully.")

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt

# ---------- USER: change if needed ----------
model_path = "/kaggle/working/best_model.keras"
# If you want to predict a specific image, set img_path; else it will use the first test example
img_path = "/kaggle/input/lungs-disease-dataset-4-types/Lung Disease Dataset/test/Normal/01.jpeg"
# --------------------------------------------

class_names = ["Bacterial Pneumonia", "Corona Virus Disease", "Normal", "Viral Pneumonia"]  # <-- Update to your order
IMG_SIZE = (224, 224)

# 2) Decide which image to use
if img_path is None:
    if 'fp_test' in globals() and len(fp_test) > 0:
        img_path = fp_test[0]
        print("Using first test image:", img_path)
    else:
        raise RuntimeError("No img_path provided and fp_test not found or empty. Set img_path manually.")

# 3) Preprocess single image using same steps as load_and_preprocess
def preprocess_single_image(path):
    raw = tf.io.read_file(path)
    img = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.convert_image_dtype(img, tf.float32)             # 0..1
    img = tf.image.resize(img, IMG_SIZE)                            # use same IMG_SIZE
    img_for_model = mobilenet_preprocess(img * 255.0)               # your loader used this
    return img_for_model, img  # return model-ready image and original resized (0..1) for display

img_model_ready, img_display = preprocess_single_image(img_path)
batch = tf.expand_dims(img_model_ready, axis=0)  # shape (1, H, W, C)

# 4) Predict
raw_pred = model.predict(batch, verbose=0)

# 5) Convert raw_pred to probability vector for the single sample
if raw_pred.ndim == 2:
    probs = tf.nn.softmax(raw_pred[0]).numpy()
elif raw_pred.ndim == 1:
    # e.g. (num_classes,) already
    probs = tf.nn.softmax(raw_pred).numpy() if not np.isclose(np.sum(raw_pred), 1.0) else raw_pred
else:
    # fallback: flatten first sample
    probs = tf.nn.softmax(raw_pred.reshape(raw_pred.shape[0], -1)[0]).numpy()

pred_idx = int(np.argmax(probs))
pred_conf = float(np.max(probs))

# 6) Print the predicted class and all probabilities
print("\nPrediction Result")
print("-----------------")
name = class_names[pred_idx] if pred_idx < len(class_names) else f"class_{pred_idx}"
print(f"Predicted class index: {pred_idx}")
print(f"Predicted class name : {name}")
print(f"Confidence (top)     : {pred_conf:.4f}\n")

print("All class probabilities:")
for i, p in enumerate(probs):
    nm = class_names[i] if i < len(class_names) else f"class_{i}"
    print(f"  {i:02d}. {nm:30s} : {p:.4f}")

# 7) Display the image with predicted label
plt.figure(figsize=(5,5))
plt.imshow(img_display.numpy())
plt.axis("off")
plt.title(f"{name} ({pred_conf:.2%})")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
# import mobilenet_preprocess again so this cell can use it
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess


# Compute confusion matrix based on unique labels seen in y_true/y_pred
unique_labels = np.unique(np.concatenate([y_true, y_pred]))
cm = confusion_matrix(y_true, y_pred, labels=unique_labels)

# If class_names is longer or shorter, adjust it safely
if len(class_names) != len(unique_labels):
    print(f"⚠️ Adjusting class names: found {len(unique_labels)} labels, but {len(class_names)} names.")
    class_labels = [class_names[i] if i < len(class_names) else f"Class_{i}" for i in range(len(unique_labels))]
else:
    class_labels = class_names

# ---------- Plot confusion matrix ----------
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()

plt.xticks(range(len(class_labels)), class_labels, rotation=45)
plt.yticks(range(len(class_labels)), class_labels)
plt.xlabel('Predicted')
plt.ylabel('True')

# Annotate each cell
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j, i, cm[i, j],
            ha='center', va='center',
            color='white' if cm[i, j] > cm.max() / 2 else 'black'
        )

plt.tight_layout()
plt.show()


In [4]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image
from glob import glob

# Folder where you manually upload X-ray images to test predictions
random_folder = "/kaggle/input/lungs-disease-dataset-4-types/Lung Disease Dataset/test/Corona Virus Disease"

# ---------- Check for folder and predict ----------
if not os.path.exists(random_folder):
    print(f"⚠️ Folder not found: {random_folder}")
    print("👉 Please create the folder and upload some X-ray images to test predictions.")
else:
    imgs = glob(os.path.join(random_folder, '*'))
    if len(imgs) == 0:
        print("⚠️ No images found in /kaggle/working/random_images folder.")
        print("👉 Please upload at least one image file (e.g., .jpg or .png).")
    else:
        # Pick a random image
        sample = random.choice(imgs)
        print('🔍 Predicting for:', sample)

        # Load and preprocess image
        img = image.load_img(sample, target_size=IMG_SIZE)
        x = image.img_to_array(img)
        x = np.expand_dims(x, axis=0)
        x = mobilenet_preprocess(x)

        # Run prediction
        preds = model.predict(x)
        pred_idx = np.argmax(preds, axis=1)[0]
        prob = float(np.max(preds))

        # Show prediction
        print(f"✅ Predicted class: {class_names[pred_idx]} | Probability: {prob:.3f}")

        # Display image
        plt.imshow(img, cmap='gray')
        plt.title(f"Pred: {class_names[pred_idx]} ({prob:.3f})")
        plt.axis('off')
        plt.show()


🔍 Predicting for: /kaggle/input/lungs-disease-dataset-4-types/Lung Disease Dataset/test/Corona Virus Disease/COVID19(264).jpg


NameError: name 'mobilenet_preprocess' is not defined

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image
from glob import glob

# -----------------------------------------
# CONFIG
# -----------------------------------------
test_root = "/kaggle/input/lungs-disease-dataset-4-types/Lung Disease Dataset/test/"
class_folders = os.listdir(test_root)
class_folders = [c for c in class_folders if os.path.isdir(os.path.join(test_root, c))]

# -----------------------------------------
# LOOP THROUGH EACH CLASS
# -----------------------------------------
for cls in class_folders:
    folder_path = os.path.join(test_root, cls)

    # get all images of this class
    images = glob(os.path.join(folder_path, "*"))
    if len(images) == 0:
        print(f"⚠️ No images found in {folder_path}")
        continue

    # pick one random image
    sample = random.choice(images)

    print("\n----------------------------------")
    print(f"📁 True Class: {cls}")
    print(f"🖼 Selected Image: {sample}")

    # load + preprocess
    img = image.load_img(sample, target_size=IMG_SIZE)
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = mobilenet_preprocess(x)   # use your preprocessing

    # predict
    preds = model.predict(x)
    pred_idx = np.argmax(preds, axis=1)[0]
    prob = float(np.max(preds))

    print(f"🔍 Predicted: {class_names[pred_idx]} | Confidence: {prob:.4f}")

    # show image
    plt.figure(figsize=(4,4))
    plt.imshow(img, cmap='gray')
    plt.title(f"True: {cls}\nPred: {class_names[pred_idx]} ({prob:.3f})")
    plt.axis("off")
    plt.show()


In [ ]:
# SaveF model in new Keras format
model.save('/kaggle/working/fpn_gated_model.keras')
print("✅ Model saved successfully to /kaggle/working/fpn_gated_model.keras")


In [ ]:
import json
import os

# path where you'll save the mapping
mapping_path = "/kaggle/working/class_mapping.json"

# Ensure class_names exists
try:
    class_names
except NameError:
    raise RuntimeError("class_names not found. Make sure you've loaded/created the dataset and have class_names list.")

# Build mapping: index (str or int) -> class name
# Use strings for keys (JSON-friendly) but you can also use ints.
index_to_class = {str(i): name for i, name in enumerate(class_names)}

# Save JSON
with open(mapping_path, "w") as f:
    json.dump(index_to_class, f, indent=2, ensure_ascii=False)

print(f"✅ Saved class mapping to: {mapping_path}")
print(index_to_class)


In [10]:
# ============================================
# Gradio App for Your Trained MobileNet Model
# ============================================

!pip install -q gradio

import json
import numpy as np
import tensorflow as tf
import gradio as gr

from PIL import Image
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

# -------------------------------------------------
# Paths
# -------------------------------------------------
MODEL_PATH = "/kaggle/working/best_model.keras"
CLASS_MAPPING_PATH = "/kaggle/working/class_mapping.json"

# -------------------------------------------------
# Load Model
# -------------------------------------------------
model = tf.keras.models.load_model(MODEL_PATH)
print("✅ Model Loaded Successfully")

# -------------------------------------------------
# Load Class Mapping
# -------------------------------------------------
with open(CLASS_MAPPING_PATH, "r") as f:
    class_mapping = json.load(f)

# Convert {"COVID":0,"Normal":1,...} -> ["COVID","Normal",...]
class_names = [None] * len(class_mapping)
for name, idx in class_mapping.items():
    class_names[int(idx)] = name

print("Classes:", class_names)

# -------------------------------------------------
# Image Size
# -------------------------------------------------
IMG_SIZE = model.input_shape[1:3]
print("Input Size:", IMG_SIZE)

# -------------------------------------------------
# Prediction Function
# -------------------------------------------------
def predict(img):

    img = img.convert("RGB")
    img = img.resize(IMG_SIZE)

    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)

    # Same preprocessing used during training
    x = mobilenet_preprocess(x)

    preds = model.predict(x, verbose=0)[0]

    return {
        class_names[i]: float(preds[i])
        for i in range(len(class_names))
    }

# -------------------------------------------------
# Gradio Interface
# -------------------------------------------------
demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil", label="Upload Chest X-ray"),
    outputs=gr.Label(num_top_classes=len(class_names), label="Prediction"),
    title="🫁 Lung Disease Classifier",
    description="Upload a chest X-ray image. The model predicts the disease class along with confidence scores."
)

# -------------------------------------------------
# Launch
# -------------------------------------------------
demo.launch(share=True, debug=True)

TypeError: <class 'keras.src.models.functional.Functional'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras.src.models.functional', 'class_name': 'Functional', 'config': {}, 'registered_name': 'Functional', 'build_config': {'input_shape': None}, 'compile_config': {'optimizer': {'module': 'keras.optimizers', 'class_name': 'Adam', 'config': {'name': 'adam', 'learning_rate': 9.999999747378752e-05, 'weight_decay': None, 'clipnorm': None, 'global_clipnorm': None, 'clipvalue': None, 'use_ema': False, 'ema_momentum': 0.99, 'ema_overwrite_frequency': None, 'loss_scale_factor': None, 'gradient_accumulation_steps': None, 'beta_1': 0.9, 'beta_2': 0.999, 'epsilon': 1e-07, 'amsgrad': False}, 'registered_name': None}, 'loss': 'categorical_crossentropy', 'loss_weights': None, 'metrics': ['accuracy'], 'weighted